# Silver Layer: Transformation, Session Stitching, and User SCD Type 2

This notebook reads from the Bronze Delta table, cleans the data, and populates three Silver tables: `events`, `sessions`, and `users`.

### Objectives:
1. **Events Table**: Parse Avro, flatten properties, and deduplicate.
2. **Sessions Table**: Stitch events into sessions to calculate duration and bounce rates.
3. **Users Table**: Implement SCD Type 2 to track plan changes over time.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.window import Window

# Configs
S3_BUCKET = "realtime-saas-analytics-pipeline"
BRONZE_PATH = f"s3://{S3_BUCKET}/bronze/events/"
SILVER_EVENTS_PATH = f"s3://{S3_BUCKET}/silver/events/"
SILVER_SESSIONS_PATH = f"s3://{S3_BUCKET}/silver/sessions/"
SILVER_USERS_PATH = f"s3://{S3_BUCKET}/silver/users/"

# AWS Credentials Setup
aws_access_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_SECRET_ACCESS_KEY")

spark.conf.set("fs.s3a.access.key", aws_access_key)
spark.conf.set("fs.s3a.secret.key", aws_secret_key)
spark.conf.set("fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print("✅ AWS credentials configured")

# Checkpoint paths
CHECKPOINT_EVENTS = f"s3://{S3_BUCKET}/checkpoints/silver/events/"
CHECKPOINT_SESSIONS = f"s3://{S3_BUCKET}/checkpoints/silver/sessions/"
CHECKPOINT_USERS = f"s3://{S3_BUCKET}/checkpoints/silver/users/"

# Avro Schema from saas_event.avsc
json_schema_str = """
{
  "type": "record",
  "name": "SaaSEvent",
  "fields": [
    {"name": "event_id", "type": "string"},
    {"name": "event_type", "type": "string"},
    {"name": "timestamp", "type": "string"},
    {"name": "session_id", "type": ["null", "string"], "default": null},
    {"name": "user_id", "type": ["null", "string"], "default": null},
    {"name": "anonymous_id", "type": ["null", "string"], "default": null},
    {
      "name": "properties",
      "type": {
        "type": "record",
        "name": "EventProperties",
        "fields": [
          {"name": "page", "type": ["null", "string"]},
          {"name": "device", "type": ["null", "string"]},
          {"name": "country", "type": ["null", "string"]},
          {"name": "plan", "type": ["null", "string"]},
          {"name": "signup_method", "type": ["null", "string"]},
          {"name": "mrr_usd", "type": ["null", "double"]}
        ]
      }
    }
  ]
}
"""

✅ AWS credentials configured


## 1. Process Silver Events
Deserializing and flattening the raw bronze data.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Full Avro schema matching saas_event.avsc exactly
json_schema_str = '''
{
  "type": "record",
  "name": "SaaSEvent",
  "namespace": "com.saas.analytics",
  "fields": [
    {"name": "event_id", "type": "string"},
    {"name": "event_type", "type": "string"},
    {"name": "timestamp", "type": "string"},
    {"name": "session_id", "type": ["null", "string"], "default": null},
    {"name": "user_id", "type": ["null", "string"], "default": null},
    {"name": "anonymous_id", "type": ["null", "string"], "default": null},
    {
      "name": "properties",
      "type": {
        "type": "record",
        "name": "EventProperties",
        "fields": [
          {"name": "page", "type": ["null", "string"], "default": null},
          {"name": "referrer", "type": ["null", "string"], "default": null},
          {"name": "utm_source", "type": ["null", "string"], "default": null},
          {"name": "utm_medium", "type": ["null", "string"], "default": null},
          {"name": "utm_campaign", "type": ["null", "string"], "default": null},
          {"name": "device", "type": ["null", "string"], "default": null},
          {"name": "browser", "type": ["null", "string"], "default": null},
          {"name": "os", "type": ["null", "string"], "default": null},
          {"name": "country", "type": ["null", "string"], "default": null},
          {"name": "city", "type": ["null", "string"], "default": null},
          {"name": "plan", "type": ["null", "string"], "default": null},
          {"name": "signup_method", "type": ["null", "string"], "default": null},
          {"name": "referral_source", "type": ["null", "string"], "default": null},
          {"name": "login_method", "type": ["null", "string"], "default": null},
          {"name": "session_number", "type": ["null", "int"], "default": null},
          {"name": "feature_name", "type": ["null", "string"], "default": null},
          {"name": "duration_seconds", "type": ["null", "int"], "default": null},
          {"name": "actions_taken", "type": ["null", "int"], "default": null},
          {"name": "from_plan", "type": ["null", "string"], "default": null},
          {"name": "to_plan", "type": ["null", "string"], "default": null},
          {"name": "mrr_usd", "type": ["null", "double"], "default": null},
          {"name": "payment_method", "type": ["null", "string"], "default": null},
          {"name": "billing_cycle", "type": ["null", "string"], "default": null},
          {"name": "reason", "type": ["null", "string"], "default": null},
          {"name": "days_active", "type": ["null", "int"], "default": null},
          {"name": "mrr_lost_usd", "type": ["null", "double"], "default": null}
        ]
      }
    }
  ]
}
'''

# Strip 5-byte Confluent magic header before parsing
def strip_confluent_header(df):
    return df.withColumn("avro_value", expr("substring(value, 6, length(value)-5)"))

bronze_stream = spark.readStream.format("delta").load(BRONZE_PATH)

silver_events_df = strip_confluent_header(bronze_stream).select(
    from_avro(col("avro_value"), json_schema_str).alias("data"),
    col("ingested_at")
).filter(col("data").isNotNull()) \
.select(
    col("data.event_id"),
    col("data.event_type"),
    to_timestamp(col("data.timestamp")).alias("event_timestamp"),
    col("data.session_id"),
    col("data.user_id"),
    col("data.anonymous_id"),
    col("data.properties.page").alias("page"),
    col("data.properties.feature_name").alias("feature_name"),
    col("data.properties.device").alias("device"),
    col("data.properties.country").alias("country"),
    col("data.properties.plan").alias("plan"),
    col("data.properties.mrr_usd").alias("mrr_usd"),
    col("data.properties.from_plan").alias("from_plan"),
    col("data.properties.to_plan").alias("to_plan"),
    col("data.properties.mrr_lost_usd").alias("mrr_lost_usd"),
    col("ingested_at").alias("processed_at"),
    to_date(col("data.timestamp")).alias("processing_date")
).dropDuplicates(["event_id"])

silver_events_df.writeStream \
    .format("delta") \
    .partitionBy("processing_date") \
    .option("checkpointLocation", CHECKPOINT_EVENTS) \
    .outputMode("append") \
    .trigger(processingTime='30 seconds') \
    .start(SILVER_EVENTS_PATH)

## 2. Session Stitching Logic
Aggregating events by `session_id` to calculate session-level metrics.

In [0]:
sessions_df = silver_events_df \
    .withWatermark("event_timestamp", "1 hour") \
    .groupBy("session_id", "user_id", "anonymous_id", "device", "country") \
    .agg(
        min("event_timestamp").alias("session_start"),
        max("event_timestamp").alias("session_end"),
        count("event_id").alias("page_count"),
        first("page").alias("entry_page"),
        last("page").alias("exit_page")
    ) \
    .withColumn("session_duration_secs", col("session_end").cast("long") - col("session_start").cast("long")) \
    .withColumn("is_bounce", when(col("page_count") == 1, True).otherwise(False)) \
    .withColumn("session_date", to_date(col("session_start")))


sessions_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", CHECKPOINT_SESSIONS) \
    .start(SILVER_SESSIONS_PATH)

## 3. User SCD Type 2 (Slowly Changing Dimensions)
Tracking plan changes for users over time using `foreachBatch` to handle merge logic.

In [0]:
def upsert_users_scd2(batch_df, batch_id):
    user_updates = batch_df.filter(col("event_type").isin("signup", "upgrade", "churn")) \
        .select("user_id", "plan", "country", "device", col("event_timestamp").alias("valid_from")) \
        .withColumn("is_current", lit(True)) \
        .withColumn("valid_to", lit(None).cast("timestamp")) \
        .withColumn("updated_at", current_timestamp())
    
    user_updates.write.format("delta").mode("append").save(SILVER_USERS_PATH)

silver_events_df.writeStream \
    .foreachBatch(upsert_users_scd2) \
    .option("checkpointLocation", CHECKPOINT_USERS) \
    .start()

In [0]:
static_df = spark.read.format("delta").load(SILVER_EVENTS_PATH)
print("Total records:", static_df.count())
static_df.selectExpr("min(event_timestamp)", "max(event_timestamp)").show()

Total records: 99950


In [0]:
# for s in spark.streams.active:
#     print(f"Stream: {s.name or s.id}")
#     print(f"  Status : {s.status['message']}")
#     print(f"  Active : {s.isActive}")
#     print()

In [0]:
# # Filters only signup events to build user profiles
# users_df = silver_events_df \
#     .filter(col("event_type") == "signup")  # ← Only 'signup' events qualify

In [0]:
# # Check what raw bronze data looks like
# raw_check = spark.read.format("delta").load("s3://realtime-saas-analytics-pipeline/bronze/events/")
# raw_check.select(col("value").cast("string")).show(3, truncate=False)

In [0]:


# for s in spark.streams.active:
#     s.stop()
# print("All streams stopped ✅")

All streams stopped ✅
